# 15N |error| on the per-frame Ramachandran map (buried vs exposed coil)

Lines: frame density (50 / 80 / 95% highest-density regions). Colour: time-weighted mean |error| of the residues visiting each bin. Regions: ellipses of Hollingsworth & Karplus 2010, in `hk_rama_regions.py`.

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import PathPatch
from scipy.ndimage import gaussian_filter

FIG4 = "../../data/analysis/Fig4"
FIG = "Figs"
os.makedirs(FIG, exist_ok=True)
# PERFRAME_NPZ = f"{FIG4}/per_protein_analysis/abnormal_coil/md_phipsi_perframe_coil_persist90_panav10seedens.npz"
PERFRAME_NPZ = f"{FIG4}/per_protein_analysis/abnormal_coil/md_phipsi_perframe_coil_persist50_panav10seedens.npz"
PRED = f"{FIG4}/results/predictions_N_seed_ensemble_10seed.json"
GT = "../../data/rBMRB/cluster_30p/PANAV/test.json"

EXPOSURE_ORDER = ["buried", "exposed"]
BURIED_C, EXPOSED_C = "#4FA8A0", "#A8DDE3"
TITLE_C = {"buried": "#020303", "exposed": "#020303"}   # darker exposed tone so the title reads on white
# single-hue teal ramp built from the group colours: light (exposed tone) -> buried teal -> dark teal
ERR_CMAP = LinearSegmentedColormap.from_list("err_teal", ["#EAF6F7", EXPOSED_C, BURIED_C, "#2C6E68", "#123F3B"])

# Ramachandran regions: Hollingsworth & Karplus 2010 Fig. 6A ellipses (digitized, fitted in hk_rama_regions.py)
import hk_rama_regions as hk
WPD_CSV = f"{FIG4}/per_protein_analysis/abnormal_coil/hk2010_fig6a_wpd_datasets.csv"
OUTSIDE, MAX_DIST = "nearest", 30          # frames inside no ellipse -> closest region within MAX_DIST deg, else "other"
ELL = hk.fit_ellipses(hk.load_wpd(WPD_CSV))
REGION_ORDER = ["beta", "PPII", "zeta", "gamma'", "delta", "alpha", "delta'", "PPII'", "other"]
REGION_LBL = {"beta": "β", "PPII": r"P$_\mathregular{II}$", "zeta": "ζ", "gamma'": "γ′", "delta": "δ",
              "alpha": "α", "delta'": "δ′", "PPII'": r"P$_\mathregular{II}$′", "other": "other"}
HIST_BIN_DEG = 2               # 2D histogram bin width
SMOOTH_DEG = 6                 # periodic Gaussian smoothing (same kernel for density and error sum)
HDR_LEVELS = (0.95, 0.8, 0.5)
MIN_N_LABEL = 5                # legend shows a median only for regions with at least this many residues

# fontweight="bold" does not reach mathtext -> bold display labels for the basin names
plt.rcParams.update({"mathtext.fontset": "custom", "mathtext.rm": "Arial",
                     "mathtext.it": "Arial:italic", "mathtext.bf": "Arial:bold"})


def set_style(base=13, family="Arial"):
    plt.rcParams.update({
        "font.family": family, "font.size": base,
        "axes.titlesize": base + 1, "axes.labelsize": base,
        "xtick.labelsize": base - 2, "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2, "svg.fonttype": "none",
    })


def hdr_thresholds(dens, fracs):
    """Density thresholds enclosing the given probability mass (highest-density regions)."""
    v = np.sort(dens.ravel())[::-1]
    cum = np.cumsum(v) / v.sum()
    return [v[np.searchsorted(cum, f)] for f in fracs]

In [ ]:
# ================= H&K region assignment: cached per-frame labels =================
# One cache shared by hk2010_rama_regions.ipynb and coil_rama_error_map.ipynb (same per-frame NPZ, same order).
# The cache is reused only if the digitized CSV, OUTSIDE / MAX_DIST and the residue set + frame count are unchanged;
# otherwise the labels are recomputed and the cache is rewritten.
import hashlib

# HK_CACHE = f"{FIG4}/per_protein_analysis/abnormal_coil/hk2010_region_labels_coil_persist90_panav10seedens.npz"
HK_CACHE = f"{FIG4}/per_protein_analysis/abnormal_coil/hk2010_region_labels_coil_persist50_panav10seedens.npz"

def hk_labels_cached(phi, psi, ell, csv_path, outside, max_dist, entry, resnum, cache=HK_CACHE):
    """phi/psi [residue, frame] -> (lab, lab_strict, dist), each [residue, frame]; read from / written to `cache`."""
    entry, resnum = np.asarray(entry).astype(str), np.asarray(resnum).astype(int)   # plain dtypes (no pickling)
    key = dict(csv_sha1=hashlib.sha1(open(csv_path, "rb").read()).hexdigest(), outside=str(outside),
               max_dist=-1.0 if max_dist is None else float(max_dist), shape=np.array(np.shape(phi)))
    if os.path.exists(cache):
        c = np.load(cache, allow_pickle=False)
        same = (str(c["csv_sha1"]) == key["csv_sha1"] and str(c["outside"]) == key["outside"]
                and float(c["max_dist"]) == key["max_dist"] and np.array_equal(c["shape"], key["shape"])
                and np.array_equal(c["entry"], entry) and np.array_equal(c["resnum"], resnum))
        if same:
            names = c["names"]
            print(f"loaded region labels from {os.path.basename(cache)}")
            return names[c["lab"]], names[c["lab_strict"]], c["dist"]
        print("region-label cache is stale (CSV, OUTSIDE/MAX_DIST or residue set changed) -> recomputing")
    lab_strict, _, dist = hk.assign(phi, psi, ell, outside="other")
    lab = lab_strict if outside == "other" else hk.assign(phi, psi, ell, outside=outside, max_dist=max_dist)[0]
    names = np.array(list(ell.index) + ["other"])
    code = {n: i for i, n in enumerate(names)}
    enc = lambda L: np.vectorize(code.get, otypes=[np.uint8])(L)
    np.savez_compressed(cache, lab=enc(lab), lab_strict=enc(lab_strict), dist=dist.astype(np.float32), names=names,
                        entry=entry, resnum=resnum, **key)
    print(f"wrote {os.path.basename(cache)}")
    return lab, lab_strict, dist


## Load per-frame phi/psi and the per-residue |error|

In [ ]:
z = np.load(PERFRAME_NPZ)
R = pd.DataFrame(dict(entry=z["entry"], resnum=z["resnum"], exposure=z["exposure"]))
phi_f, psi_f, owner = z["phi"].astype(float), z["psi"].astype(float), z["owner"]   # one entry per frame
n_fr = np.bincount(owner, minlength=len(R))
w = 1.0 / n_fr[owner]                                    # each residue sums to weight 1

pred, gt = json.load(open(PRED)), json.load(open(GT))
key_of = {v: k for k, v in gt["entry_ID"].items()}
R["acerr"] = [abs(float(pred["mean"][e][r - 1]) - float(gt["N"][key_of[e]][r - 1])) for e, r in zip(R.entry, R.resnum)]
res_err = R.acerr.to_numpy()

# H&K region per frame -> per-residue time fraction per region -> dominant region
# lab_f = hk.assign(phi_f, psi_f, ELL, outside=OUTSIDE, max_dist=MAX_DIST)[0]
lab_f = hk_labels_cached(phi_f.reshape(len(R), -1), psi_f.reshape(len(R), -1), ELL, WPD_CSV, OUTSIDE, MAX_DIST,
                         R.entry.to_numpy(), R.resnum.to_numpy())[0].ravel()
occ = np.zeros((len(R), len(REGION_ORDER)))
reg_idx = pd.Index(REGION_ORDER).get_indexer(lab_f)
assert (reg_idx >= 0).all(), f"regions missing from REGION_ORDER: {set(lab_f[reg_idx < 0])}"
np.add.at(occ, (owner, reg_idx), w)                     # per-residue fraction of frames per region
R["dom_region"] = np.array(REGION_ORDER)[occ.argmax(1)]

print(R.groupby("exposure").agg(n=("acerr", "size"), median_err=("acerr", "median"),
                                frames=("acerr", lambda s: int(n_fr[s.index].sum()))).to_string())

## Density and error maps

In [ ]:
edges = np.arange(-180, 180 + HIST_BIN_DEG, HIST_BIN_DEG)
ctr = (edges[:-1] + edges[1:]) / 2
CPHI, CPSI = np.meshgrid(ctr, ctr)
sig = SMOOTH_DEG / HIST_BIN_DEG

PANELS = [(e, (R.exposure == e).to_numpy()) for e in EXPOSURE_ORDER]
maps = {}
for e, rmask in PANELS:
    fm = rmask[owner]
    D, _, _ = np.histogram2d(psi_f[fm], phi_f[fm], bins=[edges, edges], weights=w[fm])               # rows = psi
    S, _, _ = np.histogram2d(psi_f[fm], phi_f[fm], bins=[edges, edges], weights=(w * res_err[owner])[fm])
    D, S = gaussian_filter(D, sig, mode="wrap"), gaussian_filter(S, sig, mode="wrap")
    E = S / np.maximum(D, 1e-300)                        # full map; clipped to the 95% HDR when drawn
    maps[e] = (D, E, hdr_thresholds(D, HDR_LEVELS))
allE = np.concatenate([E[D >= t[0]] for D, E, t in maps.values()])
norm = Normalize(*np.percentile(allE, [2, 98]).round(1))
print(f"colour scale: {norm.vmin} - {norm.vmax} ppm (2nd-98th percentile inside the 95% HDR, both panels)")

In [ ]:
from matplotlib.collections import LineCollection

def dash_segments(x, y, on=7.0, off=5.0, n_sub=4):
    """Split a polyline (data units; NaN = break) into explicit dash segments, so the dash pattern is
    geometry and scales with the figure in any SVG editor (stroke-dasharray does not)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    idx = np.flatnonzero(~(np.isnan(x) | np.isnan(y)))
    segs = []
    for r in np.split(idx, np.flatnonzero(np.diff(idx) > 1) + 1):    # contiguous (unbroken) pieces
        if len(r) < 2:
            continue
        px, py = x[r], y[r]
        s = np.r_[0, np.cumsum(np.hypot(np.diff(px), np.diff(py)))]  # arc length along the piece (deg)
        for s0 in np.arange(0, s[-1], on + off):
            t = np.linspace(s0, min(s0 + on, s[-1]), n_sub)          # n_sub points per dash follow the curve
            segs.append(np.column_stack([np.interp(t, s, px), np.interp(t, s, py)]))
    return segs

## Figure

In [ ]:
REGION_EDGE = "#C8553D"        # one outline colour for all regions: warm rust, distinct from the grey contours / teal map
LABEL_DIR = {"beta": None, "PPII": 0, "zeta": 180, "gamma'": 0, "delta": 215, "alpha": 0, "delta'": 0,
             "PPII'": 180}       # label direction (deg, counter-clockwise from +phi); None = at the ellipse centre


set_style(base=18)
fig = plt.figure(figsize=(12, 8))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.05], wspace=0.14)
axs = [fig.add_subplot(gs[0])]
axs.append(fig.add_subplot(gs[1], sharey=axs[0]))
cax = fig.add_subplot(gs[2])

for ax, (e, rmask) in zip(axs, PANELS):
    D, E, (t95, t80, t50) = maps[e]
    # error colour, clipped to the smooth 95% HDR outline
    hdr = ax.contourf(CPHI, CPSI, D, levels=[t95, D.max() * 1.01], colors="none", zorder=0)
    clip = PathPatch(hdr.get_paths()[0], transform=ax.transData, fc="none", ec="none")
    ax.add_patch(clip)
    im = ax.imshow(E, extent=(-180, 180, -180, 180), origin="lower", cmap=ERR_CMAP, norm=norm,
                   interpolation="bilinear", aspect="auto", zorder=1)
    im.set_clip_path(clip)
    # frame density
    ax.contour(CPHI, CPSI, D, levels=[t95, t80, t50], colors="#333",
               linewidths=[0.8, 1.1, 1.6], linestyles=[":", "-", "-"], zorder=3)

    # H&K region outlines (wrapped-axis ellipses drawn back on standard axes; break the line where it wraps),
    # one neutral colour; each region named next to its outline, pushed outward along LABEL_DIR
    sub = R[rmask]
    for n in REGION_ORDER[:-1]:
        e_ = ELL.loc[n]
        bw = hk.boundary(e_, 361)
        bp, bs = hk.unwrap(bw[:, 0], bw[:, 1])
        jump = np.r_[False, (np.abs(np.diff(bp)) > 180) | (np.abs(np.diff(bs)) > 180)]

        ax.add_collection(LineCollection(dash_segments(np.where(jump, np.nan, bp), np.where(jump, np.nan, bs)),
                                         colors=REGION_EDGE, linewidths=1.8, capstyle="butt", zorder=5))

        if LABEL_DIR[n] is None:                          # label at the centre (no free space around the outline)
            x0, y0 = hk.unwrap(e_.phi_w, e_.psi_w)
            off = (0, 0)
        else:                                             # outline point facing LABEL_DIR, label pushed outward
            ang = np.radians(LABEL_DIR[n])
            k = np.argmax((bw[:, 0] - e_.phi_w) * np.cos(ang) + (bw[:, 1] - e_.psi_w) * np.sin(ang))
            x0, y0 = hk.unwrap(bw[k, 0], bw[k, 1])
            off = (11 * np.cos(ang), 11 * np.sin(ang))
        ax.annotate(REGION_LBL[n], (float(x0), float(y0)), xytext=off, textcoords="offset points", ha="center",
                    va="center", fontsize=16, fontweight="bold", color=REGION_EDGE, zorder=7,
                    bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.5))

    # legend = text only, one column inside the panel: median |err| (ppm) by dominant region (n >= MIN_N_LABEL)
    handles = []
    for n in REGION_ORDER[:-1]:
        v = sub.acerr[sub.dom_region == n]
        if len(v) >= MIN_N_LABEL:
            handles.append(plt.Line2D([], [], color="none", label=f"{REGION_LBL[n]}  {v.median():.2f}"))
    # narrow box in the empty right-hand strip (phi > ~120 deg), below the delta-prime island
    # ax.legend(handles=handles, title="median |err| (ppm) by dominant region", loc="upper center",
    #           bbox_to_anchor=(0.5, -0.13), ncol=len(handles), fontsize=14, title_fontsize=13, frameon=False,
    #           handlelength=0, handletextpad=0, columnspacing=1.1, borderaxespad=0)


    ax.set_title(f"{e.capitalize()} loop", color=TITLE_C[e], fontweight="bold", pad=30, fontsize=17)
    # ax.text(0.5, 1.015, f"n = {int(rmask.sum())} residues,  median |err| {sub.acerr.median():.2f} ppm",
    #         transform=ax.transAxes, ha="center", va="bottom", fontsize=15)
    print(f"{e.capitalize()} coil: n = {int(rmask.sum())} residues,  median |err| {sub.acerr.median():.2f} ppm")
    ax.text(0.5, 1.015, f"median |err| {sub.acerr.median():.2f} ppm",
            transform=ax.transAxes, ha="center", va="bottom", fontsize=15)
    ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)
    ax.set_xticks(range(-180, 181, 90)); ax.set_yticks(range(-180, 181, 90))
    ax.set_aspect("equal")
    ax.set_xlabel("φ (deg, per MD frame)")
axs[0].set_ylabel("ψ (deg, per MD frame)", labelpad=0)
for ax in axs[1:]:
    plt.setp(ax.get_yticklabels(), visible=False)
cb = fig.colorbar(ScalarMappable(norm=norm, cmap=ERR_CMAP), cax=cax)
cb.set_label(r"time-weighted mean |pred - exp| $\mathregular{^{15}}$N (ppm)", fontsize=15)
axs[-1].apply_aspect()                                   # final (aspect-equal) panel box before reading it
pos = axs[-1].get_position(); cpos = cax.get_position()
CBAR_PAD = 0.012                                        # gap between the right panel and the colour bar (figure fraction)
cax.set_position([pos.x1 + CBAR_PAD, pos.y0, cpos.width, pos.height])

# out = f"{FIG}/rachman_md_coil_perframe_error_map_buried_exposed_panav10seedens"
out = f"{FIG}/rachman_md_coil_perframe_error_map_buried_exposed_persist50_panav10seedens"
fig.savefig(f"{out}.svg", bbox_inches="tight")
fig.savefig(f"{out}.png", dpi=300, bbox_inches="tight")
print(f"saved {os.path.basename(out)}")
plt.show()

In [ ]:
set_style(base=18)
# ================= V/T are enriched in the H&K beta region, in buried AND exposed coil =================
# Run after the region-assignment cell (reuses R, occ, REGION_ORDER, gt, key_of, EXPOSURE_ORDER, BURIED_C, EXPOSED_C,
# set_style, FIG). Per residue: beta time = fraction of post-equilibration frames in the H&K beta region.
# Bars = mean beta time per group; whiskers = protein-cluster bootstrap 95% CI; bracket = V/T minus other residues,
# two-sided protein-cluster bootstrap p (proteins resampled with replacement).
B_BOOT_VT = 5000
VT = ["V", "T"]
OTHER_C = "#D9D9D9"

R["aa"] = [gt["sequence"][key_of[e]][r - 1] for e, r in zip(R.entry, R.resnum)]
R["beta_time"] = occ[:, REGION_ORDER.index("beta")]
R["is_vt"] = R.aa.isin(VT)

rng_vt = np.random.default_rng(7)
stats_vt = {}
for e in EXPOSURE_ORDER:
    d = R[R.exposure == e].reset_index(drop=True)
    v, a = d.beta_time.to_numpy(), d.is_vt.to_numpy()
    rows = {p: np.where(d.entry.values == p)[0] for p in d.entry.unique()}
    ents = np.array(list(rows))
    bs_vt, bs_ot, bs_d = [], [], []
    for _ in range(B_BOOT_VT):
        i = np.concatenate([rows[p] for p in rng_vt.choice(ents, ents.size)])
        if a[i].any() and (~a[i]).any():
            bs_vt.append(v[i][a[i]].mean()); bs_ot.append(v[i][~a[i]].mean())
            bs_d.append(bs_vt[-1] - bs_ot[-1])
    bs_d = np.array(bs_d)
    stats_vt[e] = dict(vt=v[a].mean(), ot=v[~a].mean(), n_vt=int(a.sum()), n_ot=int((~a).sum()),
                       ci_vt=np.percentile(bs_vt, [2.5, 97.5]), ci_ot=np.percentile(bs_ot, [2.5, 97.5]),
                       diff=v[a].mean() - v[~a].mean(), ci_d=np.percentile(bs_d, [2.5, 97.5]),
                       p=2 * min((bs_d <= 0).mean(), (bs_d >= 0).mean()))
    s = stats_vt[e]
    print(f"{e:8s} beta time: V/T {100 * s['vt']:.1f}% (n={s['n_vt']})  vs  others {100 * s['ot']:.1f}% (n={s['n_ot']})  "
          f"diff {100 * s['diff']:+.1f}% [{100 * s['ci_d'][0]:+.1f}, {100 * s['ci_d'][1]:+.1f}]  bootstrap p = {s['p']:.2g}")


def p_txt(p):
    """significance stars from the two-sided protein-cluster bootstrap p"""
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."


set_style(base=14)
fig, ax = plt.subplots(figsize=(4.2, 5.4))
w_ = 0.4
for i, e in enumerate(EXPOSURE_ORDER):
    s = stats_vt[e]
    for off, val, ci, n, c, lab_ in [(-w_ / 2, s["vt"], s["ci_vt"], s["n_vt"], BURIED_C if e == "buried" else EXPOSED_C, "V/T"),
                                     (w_ / 2, s["ot"], s["ci_ot"], s["n_ot"], OTHER_C, "others")]:
        ax.bar(i + off, 100 * val, width=w_, color=c, edgecolor="#444", lw=0.8, zorder=2)
        ax.errorbar(i + off, 100 * val, yerr=[[100 * (val - ci[0])], [100 * (ci[1] - val)]], fmt="none",
                    ecolor="#444", elinewidth=1.2, capsize=4, zorder=3)
        ax.text(i + off, 1.5, f"{lab_}", ha="center", va="bottom", color="#222", zorder=4)
    y_br = 100 * max(s["ci_vt"][1], s["ci_ot"][1]) + 3
    ax.plot([i - w_ / 2, i - w_ / 2, i + w_ / 2, i + w_ / 2], [y_br - 1.2, y_br, y_br, y_br - 1.2], color="k", lw=1)
    ax.text(i, y_br + 0.6, p_txt(s["p"]), ha="center", va="bottom")
ax.set_xticks(range(len(EXPOSURE_ORDER)))
ax.set_xticklabels([f"{e.capitalize()} loop" for e in EXPOSURE_ORDER])
ax.set_xlim(-0.6, len(EXPOSURE_ORDER) - 0.4)
ax.set_ylim(0, 100 * max(max(s["ci_vt"][1], s["ci_ot"][1]) for s in stats_vt.values()) + 12)
ax.set_ylabel("time in β region (%)")
# ax.set_title("V/T spend more time in the β region", loc="left")
for s_ in ("top", "right"):
    ax.spines[s_].set_visible(False)

out = f"{FIG}/coil_VT_beta_time_hk2010_buried_exposed_{os.path.basename(PERFRAME_NPZ).split('_coil_')[1].split('_')[0]}_panav10seedens"
fig.savefig(f"{out}.svg", bbox_inches="tight")
fig.savefig(f"{out}.png", dpi=300, bbox_inches="tight")
print(f"saved {os.path.basename(out)}")
plt.show()


In [ ]:
# ================= time-weighted median |error| per H&K region: buried vs exposed coil =================
# Run after the region-assignment cell (reuses R, occ, REGION_ORDER, REGION_LBL, EXPOSURE_ORDER, BURIED_C, EXPOSED_C,
# set_style, FIG, PERFRAME_NPZ). Same weighting as the error map: every residue enters every region it visits,
# weighted by the fraction of its post-equilibration frames spent there (occ[i, region]); so a residue that is
# 60% beta / 40% PII adds 0.6 of its |error| distribution to beta and 0.4 to PII (no "dominant region" cut).
# Point = weighted median |error|; whiskers = protein-cluster bootstrap 95% CI; stars = buried - exposed,
# two-sided protein-cluster bootstrap p. Regions shown only if both groups carry >= MIN_W residue-equivalents.
B_BOOT_W = 2000
MIN_W = 5.0                     # minimum summed weight (residue-equivalents) per group to show a region


def wmedian(x, w):
    """weighted median: smallest x whose cumulative weight reaches half of the total"""
    o = np.argsort(x)
    cw = np.cumsum(w[o])
    return x[o][np.searchsorted(cw, 0.5 * cw[-1])]


def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."

set_style(base=16)

err_all = R.acerr.to_numpy()
regions = [n for n in REGION_ORDER if n != "other"]
rows_w = []
rng_w = np.random.default_rng(11)
grp = {e: (R.exposure == e).to_numpy() for e in EXPOSURE_ORDER}
ents_all = R.entry.unique()
rows_of_ent = {p: np.where(R.entry.values == p)[0] for p in ents_all}
boot_idx = [np.concatenate([rows_of_ent[p] for p in rng_w.choice(ents_all, ents_all.size)]) for _ in range(B_BOOT_W)]
for n in regions:
    wcol = occ[:, REGION_ORDER.index(n)]
    rec = dict(region=n)
    for e in EXPOSURE_ORDER:
        m = grp[e] & (wcol > 0)
        rec[f"w_{e}"] = wcol[m].sum()
        rec[f"med_{e}"] = wmedian(err_all[m], wcol[m]) if m.any() else np.nan
    if min(rec["w_buried"], rec["w_exposed"]) >= MIN_W:
        bs = {e: [] for e in EXPOSURE_ORDER}
        for i in boot_idx:                                         # same protein resample for both groups
            for e in EXPOSURE_ORDER:
                m = grp[e][i] & (wcol[i] > 0)
                bs[e].append(wmedian(err_all[i][m], wcol[i][m]) if m.any() else np.nan)
        b_, x_ = np.array(bs["buried"]), np.array(bs["exposed"])
        ok = np.isfinite(b_) & np.isfinite(x_)
        d_ = b_[ok] - x_[ok]
        for e, arr in [("buried", b_), ("exposed", x_)]:
            rec[f"lo_{e}"], rec[f"hi_{e}"] = np.nanpercentile(arr, [2.5, 97.5])
        rec["diff"] = rec["med_buried"] - rec["med_exposed"]
        rec["diff_lo"], rec["diff_hi"] = np.percentile(d_, [2.5, 97.5])
        rec["p"] = 2 * min((d_ <= 0).mean(), (d_ >= 0).mean())
    rows_w.append(rec)
WM = pd.DataFrame(rows_w).set_index("region")
pd.set_option("display.width", 220)
print("time-weighted median |error| per region (w = summed time weight, residue-equivalents):")
print(WM.round(2).rename(index=REGION_LBL).to_string())

# ---------- figure: one pair of points per region ----------
show = [n for n in regions if np.isfinite(WM.loc[n].get("p", np.nan))]
fig, ax = plt.subplots(figsize=(1.15 * len(show) + 1.6, 5.2))
x = np.arange(len(show)); dx = 0.17
for off, e, c in [(-dx, "buried", BURIED_C), (dx, "exposed", EXPOSED_C)]:
    y = WM.loc[show, f"med_{e}"].to_numpy()
    lo, hi = WM.loc[show, f"lo_{e}"].to_numpy(), WM.loc[show, f"hi_{e}"].to_numpy()
    ax.errorbar(x + off, y, yerr=[y - lo, hi - y], fmt="o", ms=9, color=c, mec="k", mew=1, ecolor="#444",
                elinewidth=1.2, capsize=3, zorder=3, label=f"{e.capitalize()} coil")
top = np.nanmax(WM.loc[show, ["hi_buried", "hi_exposed"]].to_numpy())
for xi, n in zip(x, show):
    y_s = np.nanmax(WM.loc[n, ["hi_buried", "hi_exposed"]].to_numpy()) + 0.06 * top
    ax.text(xi, y_s, stars(WM.loc[n, "p"]), ha="center", va="bottom", fontsize=15)
ax.set_xticks(x)
ax.set_xticklabels([REGION_LBL[n] for n in show], fontsize=15)
ax.set_xlim(-0.6, len(show) - 0.4)
ax.set_ylim(0, top * 1.22)
ax.set_ylabel(r"time-weighted median |pred - exp| $\mathregular{^{15}}$N (ppm)", fontsize=15)
ax.set_xlabel("Ramachandran region")
ax.legend(frameon=False, loc="upper right", fontsize=15)
for s_ in ("top", "right"):
    ax.spines[s_].set_visible(False)

tag = os.path.basename(PERFRAME_NPZ).split("_coil_")[1].split("_")[0]
out = f"{FIG}/coil_region_weighted_median_err_hk2010_buried_exposed_{tag}_panav10seedens"
fig.savefig(f"{out}.svg", bbox_inches="tight")
fig.savefig(f"{out}.png", dpi=300, bbox_inches="tight")
print(f"saved {os.path.basename(out)}")
plt.show()


## Table behind the legends

In [ ]:
tab_err = pd.concat({e: R[m].groupby("dom_region").acerr.agg(["median", "size"]) for e, m in PANELS}, axis=1)
print(f"median |error| (n) by dominant H&K region; legend shows a median only for n >= {MIN_N_LABEL}")
print(tab_err.reindex(REGION_ORDER).rename(index=REGION_LBL).round(2).to_string())
print("\nmean fraction of MD time per region (%):")
print((100 * pd.DataFrame(occ, columns=REGION_ORDER).groupby(R.exposure.values).mean().T).round(1).to_string())